# Qwen3.5-0.8B LoRA Fine-Tuning — Kaggle (T4 x2)

Fine-tune Qwen3.5-0.8B on any of the 30 datasets published under
`michaelowusuntim6` on Hugging Face.

**Hardware (Kaggle free tier):** T4 x2, 32 GB VRAM total, **bf16 supported on
each T4**. Quota: **30 h/week GPU** (20 h/week TPU).
**Estimated 1 epoch on 10k records: ~10 minutes** (about 15 minutes of quota
including the install and tokenisation cells).

bf16 LoRA for Qwen3.5-0.8B fits comfortably in a single T4 (~3 GB), so
`load_in_4bit=False`. `device_map="auto"` uses both T4s when present, which is
what lets the batch size go to 2.


## Quota awareness

Check remaining accelerators with `kaggle quota` in a terminal, or from a
notebook cell with:

```python
!kaggle quota
```

Current quota: 30 h/week GPU. A single epoch on a 10k-record slice costs about
15 minutes, so the free tier is good for roughly 100+ epochs per week.

**Persistence:** Kaggle wipes `/kaggle/working/` after the session ends. Save
the adapter to a Kaggle Dataset (or push it to Hugging Face) if you want to
keep it.


In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps transformers trl peft accelerate bitsandbytes datasets
!pip install -q --no-deps unsloth_zoo


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

DATASET_NAME = "code-review-qwen35"  # change to any of the 30 on HF
MAX_SEQ_LEN = 8192
EXPERT_NAME = "code_review"
LIMIT = 10000          # ~10 minutes/epoch on a 10k slice
OUT_DIR = "/kaggle/working"

print("GPUs:", torch.cuda.device_count(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-0.8B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,             # auto -> bf16 on T4
    load_in_4bit=False,     # not needed: T4 x2 has plenty of VRAM
    device_map="auto",      # uses both T4s when available
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


In [ ]:
ds = load_dataset(f"michaelowusuntim6/{DATASET_NAME}", split="train")
if LIMIT:
    ds = ds.select(range(min(LIMIT, len(ds))))

def format_record(ex):
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return ex

ds = ds.map(format_record, num_proc=2)
print(len(ds), "training records")


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=2,      # larger: two T4s, bf16, no QLoRA
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        bf16=torch.cuda.is_bf16_supported(),  # True on T4
        fp16=False,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        output_dir=f"{OUT_DIR}/outputs/{EXPERT_NAME}",
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
    ),
)
trainer.train()


In [ ]:
model.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
tokenizer.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print("adapter saved to", f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print("Kaggle wipes /kaggle/working after the session - copy it to a "
      "Kaggle Dataset or push to HF to keep it.")


In [ ]:
# from huggingface_hub import login
# import os
# login(token=os.environ["HF_TOKEN"])
# model.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}-adapter")
# tokenizer.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}-adapter")


In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "Explain a NULL pointer dereference in the Linux kernel."}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
